# Treino do detector de estruturas cirúrgicas (YOLOv8, GPU)

Este notebook treina, do zero até pesos prontos, um detector de objetos que reconhece
estruturas anatômicas e instrumentos em imagens de cirurgia laparoscópica. Ele é
**autossuficiente**: não depende de nenhum outro código deste repositório além do
arquivo `finetune.py` que está nesta mesma pasta — só usa a biblioteca `ultralytics`
para treinar e avaliar. O sistema em produção só carrega o peso final (`best.pt`)
para fazer inferência; ele nunca treina.

**Antes de abrir este notebook**: rode localmente
`python3 training/prepare_dataset_subset.py`, zipe `training/staging/` e suba o zip
para o seu Google Drive (ver `training/README.md` para o passo a passo completo).

**Preparar o ambiente**: no menu do Colab, Ambiente de execução -> Alterar tipo de
ambiente de execução -> GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste para o caminho onde você subiu o zip preparado no passo anterior.
DRIVE_STAGING_ZIP = '/content/drive/MyDrive/endoscapes_staging.zip'
# Tudo que este notebook produz (pesos, métricas) é salvo direto no Drive -- a
# sessão do Colab é temporária e pode cair por inatividade a qualquer momento.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/endoscapes_training_output'

In [ ]:
!pip install -q ultralytics

# Suba também o arquivo `finetune.py` (desta mesma pasta `training/`) para o
# Colab -- use o painel de arquivos à esquerda (ícone de pasta) e arraste o
# arquivo para a raiz de `/content/`, ou rode a célula abaixo se preferir
# colar o conteúdo diretamente aqui.
import sys
sys.path.insert(0, '/content')
from finetune import finetune

In [ ]:
import shutil
from pathlib import Path

staging_dir = Path('/content/dataset_staging')
shutil.unpack_archive(DRIVE_STAGING_ZIP, staging_dir)

# O dataset já vem dividido oficialmente em treino/validação/teste -- nunca
# misturar imagens entre esses grupos: imagens vizinhas de um mesmo vídeo
# cirúrgico são quase idênticas, e misturá-las inflaria artificialmente a
# métrica final (o modelo pareceria acertar imagens que na prática já viu).
train_coco = staging_dir / 'train' / 'annotation_coco.json'
train_images = staging_dir / 'train'
val_coco = staging_dir / 'val' / 'annotation_coco.json'
val_images = staging_dir / 'val'
test_coco = staging_dir / 'test' / 'annotation_coco.json'
test_images = staging_dir / 'test'

for arquivo in (train_coco, val_coco, test_coco):
    assert arquivo.is_file(), f'{arquivo} ausente -- confira o zip subido ao Drive'

## Treino

Usa o split de validação de verdade durante o treino (não o próprio treino outra
vez) e guarda o resultado direto no Drive.

In [ ]:
SEED = 42
EPOCHS = 50
IMGSZ = 640

best_weights = finetune(
    train_coco=train_coco,
    train_images=train_images,
    val_coco=val_coco,
    val_images=val_images,
    output_dir=Path(DRIVE_OUTPUT_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    seed=SEED,
)
print('Pesos treinados em:', best_weights)

In [ ]:
# Guarda o histórico de métricas por época ao lado do peso final, mais fácil de achar depois.
results_csv = best_weights.parent.parent / 'results.csv'
if results_csv.is_file():
    shutil.copy2(results_csv, Path(DRIVE_OUTPUT_DIR) / 'results.csv')
    print('Histórico de treino salvo em', Path(DRIVE_OUTPUT_DIR) / 'results.csv')

## Avaliação honesta contra o conjunto de teste (nunca visto no treino)

Usa a própria ferramenta de validação do `ultralytics` sobre o split de teste --
essa é a métrica que importa para o relatório, não o desempenho durante o treino.

In [ ]:
import json

from finetune import convert_split, coco_categories
from ultralytics import YOLO

yolo_dir = best_weights.parents[3] / 'yolo'
convert_split(test_coco, test_images, yolo_dir, 'test')

# O ultralytics exige as chaves `train`/`val` em qualquer YAML de dataset,
# mesmo quando só queremos validar -- apontamos as duas para o próprio split
# de teste; só a métrica de `split='val'` abaixo é usada de fato.
nomes_yaml = ''.join(
    f'  {idx}: {nome}\n' for idx, nome in sorted(coco_categories(test_coco).items())
)
test_dataset_yaml = yolo_dir / 'dataset_test.yaml'
test_dataset_yaml.write_text(
    f'path: {yolo_dir}\ntrain: images/test\nval: images/test\nnames:\n{nomes_yaml}',
    encoding='utf-8',
)

modelo = YOLO(str(best_weights))
# `project=`/`name=` explícitos evitam que o ultralytics grave em `runs/`
# relativo ao diretório de trabalho corrente (mesmo cuidado do treino acima).
metricas = modelo.val(
    data=str(test_dataset_yaml),
    split='val',
    project=str(Path(DRIVE_OUTPUT_DIR) / 'runs'),
    name='avaliacao_teste',
    exist_ok=True,
)

resumo = {
    'precision_media': float(metricas.box.mp),
    'recall_medio': float(metricas.box.mr),
    'mAP50': float(metricas.box.map50),
    'mAP50_95': float(metricas.box.map),
}
metrics_path = Path(DRIVE_OUTPUT_DIR) / 'metrics_test.json'
metrics_path.write_text(json.dumps(resumo, indent=2, ensure_ascii=False), encoding='utf-8')

print(resumo)
print('\nMétricas de teste salvas em', metrics_path)

## Próximos passos (fora do Colab)

1. Baixe `best.pt`, `results.csv` e `metrics_test.json` de `DRIVE_OUTPUT_DIR` para a
   pasta `models/` deste repositório, na sua máquina.
2. Preencha `models/README.md` com data, hiperparâmetros usados e as métricas de
   `metrics_test.json`.
3. Publique `best.pt` como anexo de uma nova versão ("Release") do repositório no
   GitHub.
4. Quem só quer rodar o sistema (sem retreinar) executa `make models-fetch`, que
   baixa esse peso publicado automaticamente.